# Audited DoC Brain-State Analysis (New BOLD Dataset)

This notebook walks through the corrected pipeline for the new DoC BOLD data, with parity checks against Brain-Act assumptions and subject-local `k=5` state extraction.

Primary constraints implemented:
- subject-local states are fitted independently per subject
- SC-FC coupling is subject-wise (each subject FC vs the same subject SC)
- canonical templates are optional post-hoc only (not used in primary fitting)
- non-finite BOLD subjects are excluded (not imputed)


In [ ]:
from pathlib import Path
import subprocess
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('/Users/borjan/CNRS/projects/TVBToolkit')
SCRIPT = ROOT / 'scripts' / 'brain_states_new_doc_bold_audited.py'
OUT = ROOT / 'results' / 'doc_patients_new_bold_brain_states_audited'

ROOT, SCRIPT, OUT

## 1) Run the audited pipeline

This executes the full corrected analysis on the new dataset.


In [ ]:
cmd = [
    'python3', str(SCRIPT),
    '--data-root', str(ROOT / 'data' / 'doc_patients_new_data'),
    '--reference-root', str(ROOT / 'data' / 'brain_act'),
    '--output-root', str(OUT),
    '--progress-every', '20'
]

res = subprocess.run(cmd, cwd=str(ROOT), capture_output=True, text=True)
print('Return code:', res.returncode)
print('
--- STDOUT (tail) ---')
print('
'.join(res.stdout.splitlines()[-30:]))
print('
--- STDERR (tail) ---')
print('
'.join(res.stderr.splitlines()[-30:]))

if res.returncode != 0:
    raise RuntimeError('Pipeline run failed.')

## 2) Load core outputs


In [ ]:
tables = OUT / 'tables'
figs = OUT / 'figures'
logs = OUT / 'logs'

subject_df = pd.read_csv(tables / 'subject_level_summary.csv')
rank_df = pd.read_csv(tables / 'rank_aligned_state_metrics_long.csv')
excluded_df = pd.read_csv(tables / 'excluded_subjects_nonfinite_bold.csv')
cohort_df = pd.read_csv(tables / 'cohort_summary.csv')
load_qc_df = pd.read_csv(tables / 'subject_loading_qc.csv')
roi_qc_df = pd.read_csv(tables / 'roi_order_qc_coupling_checks.csv')

with open(logs / 'roi_reorder_decision.json', 'r') as f:
    roi_decision = json.load(f)

print('Loaded subjects (raw):', load_qc_df.shape[0])
print('Excluded (non-finite BOLD):', excluded_df.shape[0])
print('Analyzed:', subject_df.shape[0])
print('Cohort counts analyzed:', subject_df['cohort'].value_counts().to_dict())
print('ROI reorder decision:', roi_decision['applied_mode'])
print('Mean identity coupling:', round(roi_decision['mean_identity'], 4))
print('Mean reorder_fc coupling:', round(roi_decision['mean_reorder_fc'], 4))

## 3) Demographic / structure checks


In [ ]:
print('Loaded by cohort:', load_qc_df.groupby('cohort').size().to_dict())
print('Loaded by cohort/sedation:', load_qc_df.groupby(['cohort','sedation']).size().to_dict())
print('Loaded by cohort/stage:', load_qc_df.groupby(['cohort','stage']).size().to_dict())
print('Unique loaded subject IDs:', load_qc_df['subject_id'].nunique())

if not excluded_df.empty:
    print('
Excluded by cohort:', excluded_df.groupby('cohort').size().to_dict())
    print('Excluded detail:')
    display(excluded_df)


## 4) Subject-level SC-FC coupling sanity


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5.0))
for cohort, d in roi_qc_df.groupby('cohort'):
    ax.scatter(d['coupling_identity'], d['coupling_reorder_fc'], s=18, alpha=0.7, label=cohort.upper())
mn = min(roi_qc_df['coupling_identity'].min(), roi_qc_df['coupling_reorder_fc'].min())
mx = max(roi_qc_df['coupling_identity'].max(), roi_qc_df['coupling_reorder_fc'].max())
ax.plot([mn,mx],[mn,mx],'k--',lw=1)
ax.set_xlabel('Identity ROI order')
ax.set_ylabel('FC reordered to symmetric AAL90')
ax.set_title('Static FC-SC coupling parity check')
ax.legend()
plt.show()

## 5) Occupancy and SC-coupling (subject-local states)

Primary analysis output uses local states and within-subject rank alignment (by local SC-coupling), not template-forced bins.


In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 5.0))
for cohort, d in rank_df.groupby('cohort'):
    ax.scatter(d['sfc'], d['occupancy'], s=14, alpha=0.35, label=cohort.upper())
ax.set_xlabel('State SC-coupling (Pearson r)')
ax.set_ylabel('State occupancy (subject-local)')
ax.set_title('Occupancy vs SC-coupling (continuous subject-level spread)')
ax.legend(ncol=2)
plt.show()

## 6) Cohort summary table


In [ ]:
display(cohort_df)


## 7) Generated publication figures and outputs


In [ ]:
print('Figures:')
for p in sorted(figs.glob('*')):
    print(' -', p)

print('
Tables:')
for p in sorted(tables.glob('*')):
    print(' -', p)


## 8) Interpretation notes (concise)

- The dominant technical mismatch was ROI-order parity between BOLD-derived FC and SC; applying the symmetric AAL90 reorder on the BOLD/FC side materially increased subject-level FC-SC coupling.
- Subjects with non-finite BOLD were excluded exactly as requested.
- State fitting remained subject-local (`k=5`) for all analyzed subjects.
- Cross-subject comparability used post-hoc within-subject rank alignment by SC-coupling, avoiding template enforcement during fitting.
